# EEE_project -> YOLOv9 vehicle detection (15 classes)

Reproduces arXiv:2410.08230 (YOLOv9-C, mAP@0.5 = 0.934) on the BD vehicle dataset.

Pipeline: fetch raw VOC data -> audit -> convert to YOLO -> verify -> train -> export ONNX.

The raw 7.48 GB dataset is pulled straight from Mendeley (the Kaggle mirrors are
unusable: one is a 10-class cut missing boat/horsecart/launch/tractor/
wheelbarrow, the other is classification-only with no boxes). **Internet must be ON.**

Everything bulky lives in `/kaggle/temp` (ephemeral) and is deleted as soon as it
is consumed; only weights and reports land in `/kaggle/working`.

In [ ]:
import os, subprocess, sys, glob, shutil, zipfile, time
from pathlib import Path

gpu = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE -- enable the T4 accelerator in the right-hand panel")

TMP  = Path("/kaggle/temp"); TMP.mkdir(parents=True, exist_ok=True)
RAW  = TMP / "raw"
YOLO = TMP / "eee_project_yolo"
WORK = Path("/kaggle/working"); WORK.mkdir(exist_ok=True)
ZIP  = TMP / "eee_project_data.zip"
URL  = "https://data.mendeley.com/public-api/zip/pwyyg8zmk5/download/2"

def disk_free_gb():
    st = os.statvfs("/kaggle/temp"); return st.f_bavail * st.f_frsize / 1e9

if not RAW.exists():
    print(f"free disk: {disk_free_gb():.1f} GB -- downloading 7.48 GB ...", flush=True)
    t0 = time.time()
    rc = subprocess.run(["curl","-L","--fail","--retry","3","--retry-delay","5",
                         "-o", str(ZIP), URL]).returncode
    assert rc == 0, f"download failed rc={rc}"
    print(f"downloaded {ZIP.stat().st_size/1e9:.2f} GB in {(time.time()-t0)/60:.1f} min", flush=True)

    RAW.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(RAW)
    ZIP.unlink()                      # reclaim 7.5 GB immediately
    print("extracted; zip deleted. free disk:", f"{disk_free_gb():.1f} GB")

# The zip may nest everything one level deep -- descend to the real root.
kids = [d for d in RAW.iterdir() if d.is_dir()]
if len(kids) == 1 and not any(RAW.glob("*.xml")):
    RAW = kids[0]
print("RAW =", RAW)
print("folders:", sorted(d.name for d in RAW.iterdir() if d.is_dir())[:20])
print("xml files found:", sum(1 for _ in RAW.rglob("*.xml")))

In [ ]:
# Ultralytics brings its own torch; Kaggle images already have a CUDA torch.
!pip install -q ultralytics
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1. Audit — print the real label vocabulary before trusting any class map

In [ ]:
%%writefile audit_dataset.py
#!/usr/bin/env python3
"""Read-only audit of the BD vehicle Pascal VOC dataset.

Run this BEFORE converting. Its most important output is a histogram of every
raw <object><name> string in the corpus: the class map in voc_to_yolo.py must be
built against that reality, not against the Mendeley folder names (which read
"Motorbike"/"CNG"/"Easy-bike" while the XML actually says "bike"/"cng"/"easybike").

Writes nothing. Exits 0 always -- this is a report, not a gate.
"""
from __future__ import annotations

import argparse
import collections
import os
import sys
import xml.etree.ElementTree as ET
from pathlib import Path

try:
    from PIL import Image
except ImportError:  # pillow is present on Kaggle; keep the audit usable without it
    Image = None

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}


def find_pairs(root: Path):
    """Index images and annotations by (directory, stem), pairing within a directory.

    Pairing per-directory rather than globally matters because stems repeat across
    class folders in this dataset (every folder has its own "(1)", "(2)", ...).
    """
    imgs: dict[tuple[str, str], Path] = {}
    xmls: dict[tuple[str, str], Path] = {}
    for dirpath, _dirnames, filenames in os.walk(root):
        for fn in filenames:
            p = Path(dirpath) / fn
            ext = p.suffix.lower()
            if ext in IMG_EXT:
                imgs[(dirpath, p.stem)] = p
            elif ext == ".xml":
                xmls[(dirpath, p.stem)] = p
    both = imgs.keys() & xmls.keys()
    return (
        sorted((imgs[k], xmls[k]) for k in both),
        sorted(imgs[k] for k in imgs.keys() - both),
        sorted(xmls[k] for k in xmls.keys() - both),
    )


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("root", type=Path, help="dataset root to walk")
    args = ap.parse_args()

    if not args.root.exists():
        print(f"FATAL: {args.root} does not exist", file=sys.stderr)
        return 1

    paired, img_only, xml_only = find_pairs(args.root)
    print("=" * 70)
    print(f"ROOT: {args.root}")
    print(f"paired image+xml : {len(paired)}")
    print(f"image without xml: {len(img_only)}")
    print(f"xml without image: {len(xml_only)}")

    print("\n--- top-level directories (image counts) ---")
    per_dir: collections.Counter = collections.Counter()
    for img, _ in paired:
        try:
            rel = img.relative_to(args.root).parts
        except ValueError:
            rel = img.parts
        per_dir[rel[0] if len(rel) > 1 else "<root>"] += 1
    for d, n in sorted(per_dir.items(), key=lambda kv: -kv[1]):
        print(f"  {n:6d}  {d}")

    labels: collections.Counter = collections.Counter()
    boxes_per_img: collections.Counter = collections.Counter()
    dims: collections.Counter = collections.Counter()
    difficult = truncated = 0
    size_mismatch: list[str] = []
    bad_box: list[str] = []
    unparseable: list[str] = []
    unreadable_img: list[str] = []

    for img, xml in paired:
        try:
            root_el = ET.parse(xml).getroot()
        except Exception as exc:
            unparseable.append(f"{xml}: {exc}")
            continue

        # XML-declared size
        xw = xh = None
        size_el = root_el.find("size")
        if size_el is not None:
            try:
                xw = int(float(size_el.findtext("width", "0")))
                xh = int(float(size_el.findtext("height", "0")))
            except (TypeError, ValueError):
                xw = xh = None

        # true size on disk -- the converter must trust this one, not the XML
        tw = th = None
        if Image is not None:
            try:
                with Image.open(img) as im:
                    tw, th = im.size
            except Exception as exc:
                unreadable_img.append(f"{img}: {exc}")
        if tw and xw and (tw, th) != (xw, xh):
            size_mismatch.append(f"{img.name}: xml={xw}x{xh} actual={tw}x{th}")
        if tw:
            dims[f"{tw}x{th}"] += 1

        objs = root_el.findall("object")
        boxes_per_img[len(objs)] += 1
        W, H = (tw or xw or 0), (th or xh or 0)
        for obj in objs:
            labels[(obj.findtext("name") or "<MISSING>").strip()] += 1
            if (obj.findtext("difficult") or "0").strip() == "1":
                difficult += 1
            if (obj.findtext("truncated") or "0").strip() == "1":
                truncated += 1
            bb = obj.find("bndbox")
            if bb is None:
                bad_box.append(f"{img.name}: object with no bndbox")
                continue
            try:
                x1 = float(bb.findtext("xmin")); y1 = float(bb.findtext("ymin"))
                x2 = float(bb.findtext("xmax")); y2 = float(bb.findtext("ymax"))
            except (TypeError, ValueError) as exc:
                bad_box.append(f"{img.name}: unparseable bndbox ({exc})")
                continue
            if x2 <= x1 or y2 <= y1:
                bad_box.append(f"{img.name}: degenerate {x1},{y1},{x2},{y2}")
            elif W and H and (x1 < 0 or y1 < 0 or x2 > W or y2 > H):
                bad_box.append(f"{img.name}: out of bounds {x1},{y1},{x2},{y2} in {W}x{H}")

    print("\n" + "=" * 70)
    print("RAW <object><name> HISTOGRAM  <-- build the class map from THIS")
    print("=" * 70)
    for name, n in sorted(labels.items(), key=lambda kv: -kv[1]):
        print(f"  {n:7d}  {name!r}")
    print(f"\n  distinct label strings: {len(labels)}")
    print(f"  total boxes           : {sum(labels.values())}")

    print("\n--- boxes per image ---")
    for k in sorted(boxes_per_img):
        print(f"  {boxes_per_img[k]:6d} images have {k} box(es)")

    print("\n--- 10 most common image dimensions ---")
    for d, n in dims.most_common(10):
        print(f"  {n:6d}  {d}")

    print(f"\n--- flags ---  difficult={difficult}  truncated={truncated}")
    for title, items in (
        ("XML size != actual size", size_mismatch),
        ("bad boxes", bad_box),
        ("unparseable XML", unparseable),
        ("unreadable images", unreadable_img),
        ("image without xml", [str(p) for p in img_only]),
        ("xml without image", [str(p) for p in xml_only]),
    ):
        print(f"\n--- {title}: {len(items)} ---")
        for line in items[:15]:
            print(f"    {line}")
        if len(items) > 15:
            print(f"    ... and {len(items) - 15} more")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
r = subprocess.run([sys.executable, "audit_dataset.py", str(RAW)], capture_output=True, text=True)
(WORK / "audit_report.txt").write_text(r.stdout + r.stderr)
print("\n".join(r.stdout.split("\n")[:90]))
print(f"\n[full report -> {WORK/'audit_report.txt'}]")

## 2. Convert VOC -> YOLO

Class comes from the XML `<object><name>`, never the folder name. Images are
downscaled to 1024 px (labels are normalized, so they stay valid). Near-duplicate
augmented copies are grouped by perceptual hash so they cannot straddle splits.

**A label string missing from the class map is a hard error** — a silently dropped
class is invisible accuracy loss.

In [ ]:
%%writefile voc_to_yolo.py
#!/usr/bin/env python3
"""Convert the BD vehicle dataset (Pascal VOC XML) into a YOLO detection dataset.

Produces:
    <out>/data.yaml
    <out>/images/{train,val,test}/*.jpg
    <out>/labels/{train,val,test}/*.txt      "<cls> <xc> <yc> <w> <h>", normalized 0-1
    <out>/splits.json                        group -> split, for reproducibility

Five dataset-specific hazards this handles (all verified against real files):
  1. Class comes from XML <object><name> ("bike", "cng"), NOT the folder name
     ("Motorbike", "CNG"). A folder-based converter mislabels every box.
  2. Images are 12 MP phone photos -> downscaled (labels are normalized, so
     resizing an image does not change its label file).
  3. Filenames contain spaces and parentheses: "mixed (1).jpg" -> "mixed_1.jpg".
  4. Boxes are 1-based and xmax can equal full width -> clamped, degenerates dropped.
  5. 1,791 images are augmented copies of others -> grouped by perceptual hash so
     a photo and its flipped/recoloured copies never straddle a split boundary.
"""
from __future__ import annotations

import argparse
import collections
import json
import os
import random
import re
import shutil
import sys
import xml.etree.ElementTree as ET
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
from PIL import Image

from audit_dataset import find_pairs

# Paper's class order -- keep it so per-class mAP is directly comparable.
CLASSES = [
    "bicycle", "bike", "boat", "bus", "car", "cng", "easybike", "horsecart",
    "launch", "leguna", "rickshaw", "tractor", "truck", "van", "wheelbarrow",
]
CLASS_ID = {c: i for i, c in enumerate(CLASSES)}

# Applied AFTER squashing case and separators, so "Easy-bike"/"easy bike" -> "easybike".
SYNONYMS = {
    "motorbike": "bike", "motorcycle": "bike", "motor": "bike", "bikes": "bike",
    "laguna": "leguna", "leguna": "leguna", "lagoona": "leguna",
    "lunch": "launch", "launches": "launch",
    "nouka": "boat", "boats": "boat", "nao": "boat",
    "cars": "car", "buss": "bus", "buses": "bus", "trucks": "truck",
    "cycle": "bicycle", "bycycle": "bicycle", "bicyle": "bicycle",
    "horsecarts": "horsecart", "horse": "horsecart",
    "easybikes": "easybike", "wheelbarrows": "wheelbarrow",
    "threewheeler": "cng", "autorickshaw": "cng",
}


def canon_label(raw: str) -> str:
    """'Easy-bike ' -> 'easybike'; 'Motorbike' -> 'bike'."""
    s = re.sub(r"[^a-z0-9]", "", (raw or "").strip().lower())
    return SYNONYMS.get(s, s)


def sanitize(folder: str, stem: str) -> str:
    """'Multi-class Vehicles' + 'mixed (1)' -> 'multiclassvehicles__mixed_1'."""
    f = re.sub(r"[^A-Za-z0-9]+", "", folder).lower()
    s = re.sub(r"[^A-Za-z0-9]+", "_", stem).strip("_").lower()
    return f"{f}__{s}" if f else s


def dhash(im: Image.Image, size: int = 8) -> int:
    """64-bit difference hash. Mirror-canonicalized by the caller."""
    g = im.convert("L").resize((size + 1, size), Image.LANCZOS)
    px = list(g.getdata())
    bits = 0
    for r in range(size):
        row = px[r * (size + 1):(r + 1) * (size + 1)]
        for c in range(size):
            bits = (bits << 1) | (1 if row[c] < row[c + 1] else 0)
    return bits


def canon_hash(im: Image.Image) -> int:
    """min(hash, hash-of-mirror) so a horizontal flip collides with its original."""
    return min(dhash(im), dhash(im.transpose(Image.FLIP_LEFT_RIGHT)))


# ---------------------------------------------------------------- worker

_STAGE: dict = {}


def _init(stage_img: str, stage_lbl: str, max_side: int, quality: int, keep_difficult: bool):
    _STAGE.update(img=Path(stage_img), lbl=Path(stage_lbl), max_side=max_side,
                  quality=quality, keep_difficult=keep_difficult)


def convert_one(task: tuple[str, str, str]):
    """Parse one XML + image -> staged jpg + txt. Returns a record or an error."""
    img_path, xml_path, folder = Path(task[0]), Path(task[1]), task[2]
    try:
        objs = ET.parse(xml_path).getroot().findall("object")
    except Exception as exc:
        return {"err": f"{xml_path.name}: unparseable XML ({exc})"}

    try:
        im = Image.open(img_path)
        im.load()
    except Exception as exc:
        return {"err": f"{img_path.name}: unreadable image ({exc})"}

    if im.mode != "RGB":
        im = im.convert("RGB")
    W, H = im.size  # TRUE dims -- never trust XML <size>
    if W < 2 or H < 2:
        return {"err": f"{img_path.name}: degenerate image {W}x{H}"}

    lines, classes, unknown = [], [], []
    for obj in objs:
        if not _STAGE["keep_difficult"] and (obj.findtext("difficult") or "0").strip() == "1":
            continue
        label = canon_label(obj.findtext("name"))
        if label not in CLASS_ID:
            unknown.append(obj.findtext("name"))
            continue
        bb = obj.find("bndbox")
        if bb is None:
            continue
        try:
            x1 = float(bb.findtext("xmin")); y1 = float(bb.findtext("ymin"))
            x2 = float(bb.findtext("xmax")); y2 = float(bb.findtext("ymax"))
        except (TypeError, ValueError):
            continue
        if x2 < x1:
            x1, x2 = x2, x1
        if y2 < y1:
            y1, y2 = y2, y1
        x1 = max(0.0, min(x1, W)); x2 = max(0.0, min(x2, W))
        y1 = max(0.0, min(y1, H)); y2 = max(0.0, min(y2, H))
        if (x2 - x1) < 2 or (y2 - y1) < 2:   # 1-based coords + clamping make this real
            continue
        xc, yc = ((x1 + x2) / 2) / W, ((y1 + y2) / 2) / H
        bw, bh = (x2 - x1) / W, (y2 - y1) / H
        if not (0 < bw <= 1 and 0 < bh <= 1 and 0 <= xc <= 1 and 0 <= yc <= 1):
            continue
        lines.append(f"{CLASS_ID[label]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
        classes.append(label)

    if unknown:
        return {"err": f"{xml_path.name}: UNKNOWN LABELS {sorted(set(unknown))}"}
    if not lines:
        return {"err": f"{img_path.name}: no usable boxes", "soft": True}

    h = canon_hash(im)

    max_side = _STAGE["max_side"]
    if max(W, H) > max_side:
        scale = max_side / max(W, H)
        im = im.resize((max(1, round(W * scale)), max(1, round(H * scale))), Image.LANCZOS)

    stem = sanitize(folder, img_path.stem)
    im.save(_STAGE["img"] / f"{stem}.jpg", "JPEG", quality=_STAGE["quality"], optimize=True)
    (_STAGE["lbl"] / f"{stem}.txt").write_text("\n".join(lines) + "\n")
    im.close()
    return {"stem": stem, "hash": h, "classes": classes}


# ---------------------------------------------------------------- grouping

def cluster(hashes: list[int], threshold: int) -> list[int]:
    """Union-find over perceptual hashes within `threshold` Hamming bits.

    threshold=0 groups only byte-identical hashes (exact dupes + flips).
    """
    n = len(hashes)
    parent = list(range(n))

    def find(a: int) -> int:
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a: int, b: int) -> None:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)

    # exact-hash collisions first (cheap, catches dupes and mirrored copies)
    by_hash: dict[int, list[int]] = collections.defaultdict(list)
    for i, h in enumerate(hashes):
        by_hash[h].append(i)
    for idxs in by_hash.values():
        for j in idxs[1:]:
            union(idxs[0], j)

    if threshold > 0:
        uniq = sorted(by_hash)
        rep = {h: by_hash[h][0] for h in uniq}
        arr = np.array(uniq, dtype=">u8").view(np.uint8).reshape(-1, 8)
        popc = np.unpackbits(np.arange(256, dtype=np.uint8)[:, None], axis=1).sum(1).astype(np.uint8)
        step = 512
        for s in range(0, len(arr), step):
            block = arr[s:s + step]
            dist = popc[block[:, None, :] ^ arr[None, :, :]].sum(-1)
            for bi, gj in zip(*np.where(dist <= threshold)):
                i, j = s + int(bi), int(gj)
                if i < j:
                    union(rep[uniq[i]], rep[uniq[j]])

    return [find(i) for i in range(n)]


def split_groups(records, group_of, ratios, seed):
    """Group-level split, stratified on each group's RAREST class.

    Stratifying on the rarest class is what keeps Leguna (218), Wheelbarrow (237)
    and Horse-cart (256) present in val and test.
    """
    freq: collections.Counter = collections.Counter()
    for r in records:
        freq.update(set(r["classes"]))

    members: dict[int, list] = collections.defaultdict(list)
    for r, g in zip(records, group_of):
        members[g].append(r)

    buckets: dict[str, list[int]] = collections.defaultdict(list)
    for g, rs in members.items():
        present = {c for r in rs for c in r["classes"]}
        buckets[min(present, key=lambda c: freq[c])].append(g)

    rng = random.Random(seed)
    assign: dict[int, str] = {}
    tr, va = ratios
    for cls in sorted(buckets):
        gs = sorted(buckets[cls])
        rng.shuffle(gs)
        n = len(gs)
        n_tr = max(1, round(n * tr)) if n > 2 else n
        n_va = max(1, round(n * va)) if n > 2 else 0
        if n > 2:
            n_tr = min(n_tr, n - 2)
            n_va = min(n_va, n - n_tr - 1)
        for i, g in enumerate(gs):
            assign[g] = "train" if i < n_tr else ("val" if i < n_tr + n_va else "test")
    return assign


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("root", type=Path)
    ap.add_argument("out", type=Path)
    ap.add_argument("--max-side", type=int, default=1024)
    ap.add_argument("--quality", type=int, default=90)
    ap.add_argument("--dup-threshold", type=int, default=6,
                    help="Hamming bits for near-duplicate grouping; 0 = exact+flip only")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 2)))
    ap.add_argument("--keep-difficult", action="store_true")
    ap.add_argument("--limit", type=int, default=0, help="debug: convert only N images")
    args = ap.parse_args()

    paired, img_only, xml_only = find_pairs(args.root)
    if not paired:
        print(f"FATAL: no image+xml pairs under {args.root}", file=sys.stderr)
        return 1
    if args.limit:
        paired = paired[: args.limit]
    print(f"pairs={len(paired)}  image-only={len(img_only)}  xml-only={len(xml_only)}")

    stage_img = args.out / "_stage_images"
    stage_lbl = args.out / "_stage_labels"
    for d in (stage_img, stage_lbl):
        shutil.rmtree(d, ignore_errors=True)
        d.mkdir(parents=True, exist_ok=True)

    tasks = []
    for img, xml in paired:
        try:
            folder = img.relative_to(args.root).parts[0] if len(img.relative_to(args.root).parts) > 1 else ""
        except ValueError:
            folder = img.parent.name
        tasks.append((str(img), str(xml), folder))

    records, errors, soft = [], [], []
    with ProcessPoolExecutor(
        max_workers=args.workers, initializer=_init,
        initargs=(str(stage_img), str(stage_lbl), args.max_side, args.quality, args.keep_difficult),
    ) as ex:
        for i, res in enumerate(ex.map(convert_one, tasks, chunksize=16), 1):
            if "err" in res:
                (soft if res.get("soft") else errors).append(res["err"])
            else:
                records.append(res)
            if i % 500 == 0:
                print(f"  converted {i}/{len(tasks)}", flush=True)

    print(f"\nconverted={len(records)}  skipped_no_boxes={len(soft)}  errors={len(errors)}")
    for e in errors[:20]:
        print(f"  ERR {e}")

    # An unmapped label is a silently-lost class, so refuse to continue.
    unknown = [e for e in errors if "UNKNOWN LABELS" in e]
    if unknown:
        print(f"\nFATAL: {len(unknown)} file(s) carry labels missing from the class map.", file=sys.stderr)
        print("Add them to SYNONYMS (or CLASSES) and re-run. First few:", file=sys.stderr)
        for e in unknown[:10]:
            print(f"  {e}", file=sys.stderr)
        return 2
    if not records:
        print("FATAL: nothing converted", file=sys.stderr)
        return 1

    group_of = cluster([r["hash"] for r in records], args.dup_threshold)
    n_groups = len(set(group_of))
    print(f"grouping: {len(records)} images -> {n_groups} groups "
          f"({len(records) - n_groups} near-duplicates absorbed)")

    assign = split_groups(records, group_of, (0.8, 0.1), args.seed)

    for sub in ("images", "labels"):
        for sp in ("train", "val", "test"):
            (args.out / sub / sp).mkdir(parents=True, exist_ok=True)

    per_split: collections.Counter = collections.Counter()
    per_split_cls: dict[str, collections.Counter] = collections.defaultdict(collections.Counter)
    splits_json = {}
    for r, g in zip(records, group_of):
        sp = assign[g]
        stem = r["stem"]
        shutil.move(str(stage_img / f"{stem}.jpg"), str(args.out / "images" / sp / f"{stem}.jpg"))
        shutil.move(str(stage_lbl / f"{stem}.txt"), str(args.out / "labels" / sp / f"{stem}.txt"))
        per_split[sp] += 1
        per_split_cls[sp].update(r["classes"])
        splits_json[stem] = {"split": sp, "group": int(g)}

    shutil.rmtree(stage_img, ignore_errors=True)
    shutil.rmtree(stage_lbl, ignore_errors=True)

    (args.out / "data.yaml").write_text(
        f"path: {args.out.resolve()}\n"
        "train: images/train\nval: images/val\ntest: images/test\n"
        f"nc: {len(CLASSES)}\nnames: [{', '.join(CLASSES)}]\n"
    )
    (args.out / "splits.json").write_text(json.dumps(splits_json, indent=1))

    print("\n--- images per split ---")
    for sp in ("train", "val", "test"):
        print(f"  {sp:5s} {per_split[sp]:6d}")
    print("\n--- boxes per class per split ---")
    print(f"  {'class':<12}{'train':>8}{'val':>7}{'test':>7}")
    missing = []
    for c in CLASSES:
        t, v, s = per_split_cls['train'][c], per_split_cls['val'][c], per_split_cls['test'][c]
        print(f"  {c:<12}{t:>8}{v:>7}{s:>7}")
        if v == 0 or s == 0:
            missing.append(c)
    if missing:
        print(f"\n  WARNING: absent from val or test: {missing}")
    print(f"\nwrote {args.out / 'data.yaml'}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
rc = subprocess.run([sys.executable, "voc_to_yolo.py", str(RAW), str(YOLO),
                     "--max-side", "1024", "--dup-threshold", "6", "--seed", "42"],
                    text=True).returncode
assert rc == 0, f"conversion failed rc={rc} (scroll up: likely an unmapped label)"
shutil.rmtree(RAW, ignore_errors=True)        # raw 7.5 GB no longer needed
print(open(YOLO / "data.yaml").read())

## 3. Verify — range check plus a drawn overlay (catches x/y transposes)

In [ ]:
%%writefile verify_labels.py
#!/usr/bin/env python3
"""Verify a converted YOLO dataset: numeric sanity + a visual contact sheet.

The range check catches malformed values; only the drawn overlay catches an
x/y transpose or a corner-vs-centre mistake, which are otherwise invisible.
"""
from __future__ import annotations

import argparse, random, sys
from pathlib import Path
from PIL import Image, ImageDraw

CLASSES = ["bicycle","bike","boat","bus","car","cng","easybike","horsecart",
           "launch","leguna","rickshaw","tractor","truck","van","wheelbarrow"]
PALETTE = ["#e6194b","#3cb44b","#ffe119","#4363d8","#f58231","#911eb4","#46f0f0",
           "#f032e6","#bcf60c","#fabebe","#008080","#e6beff","#9a6324","#fffac8","#800000"]


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("root", type=Path)
    ap.add_argument("--sheet", type=Path, default=Path("contact_sheet.jpg"))
    ap.add_argument("-n", type=int, default=12)
    ap.add_argument("--cols", type=int, default=4)
    ap.add_argument("--cell", type=int, default=420)
    args = ap.parse_args()

    problems, total_boxes, picks = [], 0, []
    for split in ("train", "val", "test"):
        idir, ldir = args.root / "images" / split, args.root / "labels" / split
        if not idir.exists():
            continue
        images = sorted(idir.glob("*.jpg"))
        labels = sorted(ldir.glob("*.txt"))
        istems, lstems = {p.stem for p in images}, {p.stem for p in labels}
        if istems != lstems:
            problems.append(f"{split}: {len(istems-lstems)} images w/o label, "
                            f"{len(lstems-istems)} labels w/o image")
        for lp in labels:
            for ln, line in enumerate(lp.read_text().split("\n"), 1):
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) != 5:
                    problems.append(f"{lp.name}:{ln} has {len(parts)} fields, want 5")
                    continue
                try:
                    cid = int(parts[0]); vals = [float(v) for v in parts[1:]]
                except ValueError:
                    problems.append(f"{lp.name}:{ln} non-numeric: {line!r}")
                    continue
                total_boxes += 1
                if not 0 <= cid < len(CLASSES):
                    problems.append(f"{lp.name}:{ln} class_id {cid} out of range")
                if any(not (0.0 <= v <= 1.0) for v in vals):
                    problems.append(f"{lp.name}:{ln} value outside [0,1]: {vals}")
                xc, yc, w, h = vals
                if w <= 0 or h <= 0:
                    problems.append(f"{lp.name}:{ln} non-positive w/h: {w},{h}")
                if xc - w/2 < -1e-6 or xc + w/2 > 1+1e-6 or yc - h/2 < -1e-6 or yc + h/2 > 1+1e-6:
                    problems.append(f"{lp.name}:{ln} box escapes frame: {vals}")
        picks += [(p, ldir / f"{p.stem}.txt") for p in images]

    print(f"total boxes checked: {total_boxes}")
    print(f"problems: {len(problems)}")
    for p in problems[:25]:
        print(f"  {p}")

    random.seed(0)
    random.shuffle(picks)
    picks = picks[: args.n]
    if picks:
        cols = min(args.cols, len(picks))
        rows = (len(picks) + cols - 1) // cols
        cell = args.cell
        sheet = Image.new("RGB", (cols * cell, rows * cell), "#111111")
        for i, (ip, lp) in enumerate(picks):
            with Image.open(ip) as im:
                im = im.convert("RGB")
                W, H = im.size
                d = ImageDraw.Draw(im)
                if lp.exists():
                    for line in lp.read_text().split("\n"):
                        if not line.strip():
                            continue
                        cid, xc, yc, w, h = line.split()
                        cid = int(cid); xc, yc, w, h = map(float, (xc, yc, w, h))
                        x1, y1 = (xc - w/2) * W, (yc - h/2) * H
                        x2, y2 = (xc + w/2) * W, (yc + h/2) * H
                        col = PALETTE[cid % len(PALETTE)]
                        d.rectangle([x1, y1, x2, y2], outline=col, width=max(2, W // 180))
                        d.text((x1 + 4, max(0, y1 + 2)), CLASSES[cid], fill=col)
                im.thumbnail((cell, cell), Image.LANCZOS)
                sheet.paste(im, (i % cols * cell + (cell - im.width) // 2,
                                 i // cols * cell + (cell - im.height) // 2))
        sheet.save(args.sheet, "JPEG", quality=88)
        print(f"\ncontact sheet -> {args.sheet}  ({len(picks)} images)")
    return 1 if problems else 0

if __name__ == "__main__":
    sys.exit(main())

In [ ]:
subprocess.run([sys.executable, "verify_labels.py", str(YOLO),
                "--sheet", str(WORK / "contact_sheet.jpg"), "-n", "16"], text=True)
from IPython.display import Image as IPyImage, display
display(IPyImage(str(WORK / "contact_sheet.jpg"), width=950))

## 4. Train

Paper's hyperparameters: 640 px, 40 epochs, SGD, lr0 0.01, momentum 0.85,
warmup_momentum 0.7, mixup 0.2, effective batch 32.

`batch=16` + `nbs=32` gives the paper's effective batch of 32 through gradient
accumulation on **one** GPU — batch 32 outright would OOM a 16 GB T4, and
multi-GPU DDP is unreliable inside a notebook. Training runs from a .py file
for the same reason.

`yolov9c` (the paper's model) trains first and is saved before `yolov9s`
(small, for real-time CPU inference) starts, so a failure in the second
never costs you the first.

In [ ]:
%%writefile train_yolo.py
#!/usr/bin/env python3
"""Fine-tune YOLOv9 on a YOLO-format dataset and export ONNX.

Works with either dataset in this project:
  * the raw VOC conversion produced by voc_to_yolo.py (15 classes), or
  * a pre-converted export such as archive.zip (10 classes, train/valid/test).

Two things this fixes that bite people with pre-converted exports:

  1. Roboflow writes `train: ../train/images` into a data.yaml that sits at the
     dataset ROOT. Ultralytics resolves that relative to the yaml, so `../`
     escapes the dataset and training dies with "no images found". We always
     rewrite data.yaml with absolute paths.
  2. Roboflow names the validation folder `valid`; Ultralytics expects `val`.
     Both are detected.

Paper hyperparameters (arXiv:2410.08230): 640 px, 40 epochs, SGD, lr0 0.01,
momentum 0.85, warmup_momentum 0.7, mixup 0.2, effective batch 32.
`batch=16` + `nbs=32` reaches that effective batch by gradient accumulation --
batch 32 outright OOMs a 16 GB T4.
"""
from __future__ import annotations

import argparse
import shutil
import sys
from pathlib import Path


def find_split(root: Path, *names: str) -> Path | None:
    """Locate a split's image dir under either common YOLO layout.

    Roboflow writes <root>/train/images; voc_to_yolo.py (and the Ultralytics
    docs) write <root>/images/train. Both are valid, so check both.
    """
    for n in names:
        for d in (root / n / "images", root / "images" / n):
            if d.is_dir() and any(d.iterdir()):
                return d
    return None


def read_names(root: Path) -> list[str] | None:
    """Pull class names out of an existing data.yaml without needing a yaml parser."""
    for cand in (root / "data.yaml", root / "data.yml"):
        if not cand.exists():
            continue
        for line in cand.read_text().splitlines():
            line = line.strip()
            if line.startswith("names:") and "[" in line:
                inner = line.split("[", 1)[1].rsplit("]", 1)[0]
                names = [p.strip().strip("'\"") for p in inner.split(",") if p.strip()]
                if names:
                    return names
    return None


def build_yaml(root: Path, out: Path) -> tuple[Path, list[str]]:
    train = find_split(root, "train")
    val = find_split(root, "val", "valid")
    test = find_split(root, "test")
    if train is None or val is None:
        sys.exit(f"FATAL: need train/images and val|valid/images under {root}")

    names = read_names(root)
    if names is None:
        sys.exit(f"FATAL: no class names found in {root}/data.yaml -- cannot guess them")

    lines = [
        f"path: {root.resolve()}",
        f"train: {train.resolve()}",
        f"val: {val.resolve()}",
    ]
    if test is not None:
        lines.append(f"test: {test.resolve()}")
    lines += [f"nc: {len(names)}", "names: [" + ", ".join(names) + "]"]
    out.write_text("\n".join(lines) + "\n")
    return out, names


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("root", type=Path, help="dataset root (contains train/ val|valid/ test/)")
    ap.add_argument("--models", default="yolov9c,yolov9s",
                    help="comma list; trained in order, each saved before the next starts")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--nbs", type=int, default=32, help="nominal batch -> gradient accumulation")
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--out", type=Path, default=Path("/kaggle/working"))
    ap.add_argument("--device", default="0")
    args = ap.parse_args()

    from ultralytics import YOLO

    args.out.mkdir(parents=True, exist_ok=True)
    data_yaml, names = build_yaml(args.root, args.out / "data_abs.yaml")
    print(f"dataset : {args.root}")
    print(f"classes : {len(names)} -> {names}")
    print(f"data.yaml rewritten with absolute paths -> {data_yaml}\n")

    for name in [m.strip() for m in args.models.split(",") if m.strip()]:
        print(f"\n{'=' * 60}\nTRAINING {name}\n{'=' * 60}", flush=True)
        try:
            model = YOLO(f"{name}.pt")
            model.train(
                data=str(data_yaml), epochs=args.epochs, imgsz=args.imgsz,
                batch=args.batch, nbs=args.nbs,
                optimizer="SGD", lr0=0.01, lrf=0.01,
                momentum=0.85, warmup_momentum=0.7, mixup=0.2,
                cache=False, workers=args.workers, patience=15, seed=42,
                device=args.device,
                project=str(args.out / "runs"), name=name, exist_ok=True,
            )
            best = args.out / "runs" / name / "weights" / "best.pt"
            if not best.exists():
                print(f"!! {name}: no best.pt produced", flush=True)
                continue
            shutil.copy(best, args.out / f"{name}_best.pt")

            m = YOLO(str(best))
            if "test:" in data_yaml.read_text():
                m.val(data=str(data_yaml), split="test",
                      project=str(args.out / "runs"), name=f"{name}_test", exist_ok=True)
            onnx = m.export(format="onnx", imgsz=args.imgsz, opset=12, simplify=True)
            shutil.copy(onnx, args.out / f"{name}_best.onnx")
            # Inference must know the exact class order this model was trained on;
            # the 10-class cut and the 15-class build differ.
            (args.out / f"{name}_best.classes.txt").write_text("\n".join(names) + "\n")
            print(f"SAVED {name}: {name}_best.pt + {name}_best.onnx", flush=True)
        except Exception as exc:
            print(f"!! {name} FAILED: {type(exc).__name__}: {exc}", flush=True)
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
rc = subprocess.run([sys.executable, "train_yolo.py", str(YOLO),
                     "--out", str(WORK), "--models", "yolov9c,yolov9s"], text=True).returncode
print("train_yolo.py exit code:", rc)

## 5. Results

In [ ]:
import pandas as pd
for name in ("yolov9c", "yolov9s"):
    csv = WORK / "runs" / name / "results.csv"
    if csv.exists():
        df = pd.read_csv(csv)
        df.columns = [c.strip() for c in df.columns]
        last = df.iloc[-1]
        print(f"{name}: mAP50={last.get('metrics/mAP50(B)'):.4f} "
              f"mAP50-95={last.get('metrics/mAP50-95(B)'):.4f} "
              f"P={last.get('metrics/precision(B)'):.4f} R={last.get('metrics/recall(B)'):.4f}")
print()
for f in sorted(WORK.glob('*.pt')) + sorted(WORK.glob('*.onnx')):
    print(f.name, round(f.stat().st_size/1e6, 1), "MB")